# Practice 1.1 - N-gram Language Model

Objectives:
- Build an n-gram language model at the **syllable level** for Vietnamese.
- Use the training corpus (~9,000 news articles).
- Compute **perplexity** on an unseen test set (~1,000 news articles).

Exercise Repository: https://github.com/sontungkieu/TA_NLP/tree/exercise

## Table of Contents

### Section 1: Setup
1. Import libraries and define data paths.
2. Load the train/test corpora and inspect raw samples.
3. Tokenize at the **syllable level** and add sentence boundary tokens.
4. Build the vocabulary and replace rare syllables with `<UNK>`.
5. Count n-grams for multiple orders (unigram, bigram, trigram).

### Section 2: N-gram Without Smoothing (Pure MLE)
6. Define MLE probability (no smoothing) and inspect seen vs. unseen n-grams.
7. Compute test-set perplexity with MLE.

### Section 3: N-gram With Laplace Smoothing
8. Define probability with Laplace (add-α) smoothing and inspect seen vs. unseen n-grams.
9. Compute test-set perplexity with Laplace smoothing.
10. Perform a qualitative check by suggesting the next syllable from context.

### Section 4: Qualitative Check

## Section 1: Setup

Shared infrastructure — corpus loading, tokenization, vocabulary, and n-gram counts used by both sections below.

In [ ]:
from pathlib import Path
import json
import math
from collections import Counter
import pandas as pd

data_dir = Path('.')
train_path = data_dir / 'train-Subset.txt'
test_path = data_dir / 'test-Subset.txt'

print(f'Train file found: {train_path.exists()} | Test file found: {test_path.exists()}')

In [ ]:
# Load JSON corpora
train_records = json.loads(train_path.read_text(encoding='utf-8'))
test_records = json.loads(test_path.read_text(encoding='utf-8'))

print(f'Train JSON records loaded: {len(train_records):,}')
print(f'Test JSON records loaded : {len(test_records):,}')

In [ ]:
# Parse JSON corpora and extract article text fields
def extract_text_records(records):
    texts = []
    for rec in records:
        if not isinstance(rec, dict):
            continue
        content = (rec.get('content') or '').strip()
        title = (rec.get('title') or '').strip()
        text = content if content else title
        if text:
            text = ' '.join(text.split())
            texts.append(text)
    return texts

train_lines = extract_text_records(train_records)
test_lines = extract_text_records(test_records)

print(f'Number of train articles: {len(train_lines):,}')
print(f'Number of test  articles: {len(test_lines):,}')
print('\nSample train articles:')
for i, line in enumerate(train_lines[36:40], start=1):
    print(f'{i}. {line[:180]}')

In [ ]:
# Syllable-level tokenization
PUNCT = set('!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~\n\t')

def normalize_token(token: str):
    token = token.lower()
    if token.startswith('http') or token.startswith('www.'):
        return '<url>'
    stripped = token.strip(''.join(PUNCT))
    if not stripped:
        return None
    stripped = stripped.replace(',', '')
    if not stripped:
        return None
    if all(ch.isdigit() or ch in '-./:' for ch in stripped):
        return '<num>'
    return stripped

def tokenize_syllables(line: str):
    tokens = []
    for raw in line.split():
        norm = normalize_token(raw)
        if norm:
            tokens.append(norm)
    return tokens

def add_boundaries(tokens, n):
    return ['<s>'] * (n - 1) + tokens + ['</s>']

# TODO: Create train_tokens and test_tokens from train_lines and test_lines.
############## WRITE YOUR CODE HERE ##############

print('Example tokenized sentence:', train_tokens[36][:36])

In [ ]:
# Build vocabulary from the training corpus
min_count = 2
syllable_freq = Counter(s for sent in train_tokens for s in sent)
vocab = {s for s, c in syllable_freq.items() if c >= min_count}
vocab.update({'<UNK>', '<s>', '</s>'})

print(f'Vocabulary size (after threshold): {len(vocab):,}')
print(f'Number of rare syllables below threshold: {sum(1 for c in syllable_freq.values() if c < min_count):,}')
print('Top 10 frequent syllables:', syllable_freq.most_common(10))

In [ ]:
# Map rare syllables to <UNK>
def replace_rare(tokens, vocab_set):
    return [t if t in vocab_set else '<UNK>' for t in tokens]

# TODO: Create train_tokens and test_tokens with <UNK> replacement.
############## WRITE YOUR CODE HERE ##############

print('Example sentence after <UNK> mapping:', train_tokens[36][:36])
print(f'Training sentences mapped: {len(train_tokens):,}')
print(f'Test sentences mapped    : {len(test_tokens):,}')

In [ ]:
# Count n-grams and (n-1)-gram contexts
def build_ngram_counts(tokenized_sents, n):
    """
    Count n-gram and context occurrences in a tokenized corpus.

    Args:
        tokenized_sents: list of token lists (sentences).
        n: n-gram order (1=unigram, 2=bigram, 3=trigram).

    Returns:
        ngram_counts: dict mapping n-gram tuple → count.
        context_counts: dict mapping (n-1)-gram context tuple → count.
    """
    ngram_counts = {}
    context_counts = {}
    for sent in tokenized_sents:
        # TODO: Create the boundary-augmented token sequence for this sentence.
        ############## WRITE YOUR CODE HERE ##############
        
        for i in range(len(seq) - n + 1):
            ngram = tuple(seq[i:i+n])
            context = tuple(seq[i:i+n-1]) if n > 1 else tuple()
            # TODO: Update the count dictionaries for the current n-gram and context.
            ############## WRITE YOUR CODE HERE ##############
            
    return ngram_counts, context_counts

In [ ]:
models = {}
for n in [1, 2, 3]:
    models[n] = build_ngram_counts(train_tokens, n)

print('Unigram count size:', len(models[1][0]))
print('Bigram count size :', len(models[2][0]))
print('Trigram count size:', len(models[3][0]))

print('\nTop 10 bigrams:')
for bg, c in sorted(models[2][0].items(), key=lambda x: x[1], reverse=True)[:10]:
    print(bg, '->', c)

## Section 2: N-gram Language Model Without Smoothing (Pure MLE)

The **Maximum Likelihood Estimate (MLE)** assigns zero probability to any n-gram not seen in training. This causes the perplexity to become undefined (infinite) on held-out data whenever an unseen n-gram is encountered.

In [ ]:
# N-gram probability WITHOUT smoothing (pure MLE)
def ngram_prob_mle(ngram, n, models):
    """
    Compute the MLE probability of an n-gram (no smoothing).

    Args:
        ngram: tuple of n tokens.
        n: n-gram order.
        models: dict mapping n → (ngram_counts, context_counts).

    Returns:
        Probability as a float; 0.0 for unseen n-grams or contexts.
    """
    ngram_counts, context_counts = models[n]
    context = ngram[:-1] if n > 1 else tuple()
    # TODO: Use .get() to read the n-gram/context counts, then return 0.0 for unseen cases or the MLE ratio otherwise.
    ############## WRITE YOUR CODE HERE ##############

In [ ]:
# Quick sanity check: seen vs unseen trigram
seen_trigram = tuple(add_boundaries(train_tokens[36], 3)[33:36])
unseen_trigram = ('<s>', '<s>', '__oov__')

print('Seen trigram:  ', seen_trigram)
print(f'  MLE probability: {ngram_prob_mle(seen_trigram, 3, models):.8f}')
print()
print('Unseen trigram:', unseen_trigram)
print(f'  MLE probability: {ngram_prob_mle(unseen_trigram, 3, models):.8f}  ← zero, no smoothing')

In [ ]:
# Compute perplexity on test set — MLE (no smoothing)
def perplexity_mle(tokenized_sents, n, models):
    """
    Compute test-set perplexity using MLE (no smoothing).

    Args:
        tokenized_sents: list of token lists (test sentences).
        n: n-gram order.
        models: dict mapping n → (ngram_counts, context_counts).

    Returns:
        perplexity: float (inf if any zero-probability n-gram is encountered).
        zero_count: number of n-grams assigned zero probability.
    """
    log_prob_sum = 0.0
    token_count = 0
    zero_count = 0
    for sent in tokenized_sents:
        seq = add_boundaries(sent, n)
        for i in range(n - 1, len(seq)):
            ngram = tuple(seq[i - n + 1:i + 1])
            # TODO: Compute the probability of the current n-gram under the MLE model.
            ############## WRITE YOUR CODE HERE ##############
            
            if p == 0.0:
                zero_count += 1
            else:
                log_prob_sum += math.log(p)
            token_count += 1
    if zero_count > 0:
        return float('inf'), zero_count
    # TODO: Return the finite perplexity together with the zero-count value.
    ############## WRITE YOUR CODE HERE ##############
    

rows_mle = []
for n in [1, 2, 3]:
    ppl, zeros = perplexity_mle(test_tokens, n, models)
    rows_mle.append({
        'n': n,
        'perplexity (MLE)': ppl if zeros == 0 else f'inf  ({zeros:,} zero-prob tokens)'
    })

pd.DataFrame(rows_mle).sort_values('n')

## Section 3: N-gram Language Model With Laplace Smoothing

**Laplace (add-α) smoothing** adds a small constant α to every n-gram count before normalising. This guarantees a non-zero probability for every possible n-gram, so perplexity is always finite — even for n-grams never seen in training.

In [ ]:
# N-gram probability WITH Laplace smoothing
def ngram_prob(ngram, n, models, vocab_size, alpha=1.0):
    """
    Compute the Laplace-smoothed probability of an n-gram.

    Args:
        ngram: tuple of n tokens.
        n: n-gram order.
        models: dict mapping n → (ngram_counts, context_counts).
        vocab_size: vocabulary size V (used in smoothing denominator).
        alpha: smoothing constant (default 1.0).

    Returns:
        Smoothed probability as a float.
    """
    ngram_counts, context_counts = models[n]
    context = ngram[:-1] if n > 1 else tuple()
    # TODO: Compute the Laplace-smoothed probability using the n-gram count, context count, alpha, and vocabulary size.
    ############## WRITE YOUR CODE HERE ##############

In [ ]:
# Quick sanity check: seen vs unseen trigram
seen_trigram = tuple(add_boundaries(train_tokens[36], 3)[33:36])
unseen_trigram = ('<s>', '<s>', '__oov__')

print('Seen trigram:  ', seen_trigram)
print(f'  Laplace probability: {ngram_prob(seen_trigram, 3, models, vocab_size=len(vocab)):.8f}')
print()
print('Unseen trigram:', unseen_trigram)
print(f'  Laplace probability: {ngram_prob(unseen_trigram, 3, models, vocab_size=len(vocab)):.8f}  ← small but non-zero')

In [ ]:
# Compute perplexity on test set — Laplace smoothing
def perplexity(tokenized_sents, n, models, vocab_size, alpha=1.0):
    """
    Compute test-set perplexity using Laplace smoothing.

    Args:
        tokenized_sents: list of token lists (test sentences).
        n: n-gram order.
        models: dict mapping n → (ngram_counts, context_counts).
        vocab_size: vocabulary size V (used in smoothing denominator).
        alpha: smoothing constant (default 1.0).

    Returns:
        Perplexity as a float.
    """
    log_prob_sum = 0.0
    token_count = 0
    for sent in tokenized_sents:
        seq = add_boundaries(sent, n)
        for i in range(n - 1, len(seq)):
            ngram = tuple(seq[i - n + 1:i + 1])
            # TODO: Compute the probability of the current n-gram under the smoothed model.
            ############## WRITE YOUR CODE HERE ##############

            # TODO: Update the accumulated log-probability, token count, and final perplexity.
            ############## WRITE YOUR CODE HERE ##############

rows_laplace = []
for n in [1, 2, 3]:
    ppl = perplexity(test_tokens, n, models, vocab_size=len(vocab), alpha=1.0)
    rows_laplace.append({'n': n, 'perplexity (Laplace, α=1)': round(ppl, 2)})

pd.DataFrame(rows_laplace).sort_values('n')

## Qualitative Check

In [ ]:
# Qualitative check: suggest next syllables from an in-sentence trigram context
trigram_counts, _ = models[3]

def suggest_next_syllable(context, trigram_counts, top_k=10):
    """
    Suggest likely next syllables from a two-syllable context.

    Args:
        context_2: tuple containing the two previous syllables.
        trigram_counts: dict mapping trigram tuples to counts.
        top_k: number of suggestions to return.

    Returns:
        List of (syllable, count) pairs sorted by descending count.
    """
    c1, c2 = context
    candidates = []
    for (w1, w2, w3), count in trigram_counts.items():
        if w1 == c1 and w2 == c2 and w3 not in {'<s>', '</s>'}:
            candidates.append((w3, count))
    candidates.sort(key=lambda item: item[1], reverse=True)
    return candidates[:top_k]

# Pick an in-sentence context from a reasonably long training example
example_sentence = next(sent for sent in train_tokens if len(sent) >= 12)
context_start = min(10, len(example_sentence) - 2)
example_context = tuple(example_sentence[context_start:context_start + 2])
suggestions = suggest_next_syllable(example_context, trigram_counts, top_k=10)

print('Context:', example_context)
print('Top next-syllable suggestions:')
for syllable, count in suggestions:
    print(f'{syllable:20s} {count}')